# Assistente de Iluminação Pública (rua escura) – RAG local
GS 2026.2 CitizenTech · Generative AI for Engineering

Pipeline: base de trechos com fonte → embeddings locais (Ollama) → busca top-k + limiar → LLM local com JSON, citação e recusa → agente com verificador.
Configurações comparadas: **A** LLM sem contexto · **B** RAG simples · **C** agente com verificador.

In [1]:
import os, sys, json, subprocess, pathlib, requests
import pandas as pd
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'src'))
import ruaescura as r
print('Modelo LLM:', r.LLM, '| Embeddings:', r.EMB)
print('Ollama:', requests.get(r.OLLAMA + '/api/version').json())

Modelo LLM: qwen2.5:1.5b-instruct | Embeddings: nomic-embed-text
Ollama: {'version': '0.40.15'}


## 1. Base de conhecimento (trechos com fonte)

In [2]:
T = pd.DataFrame(r.ler_jsonl(ROOT / 'base' / 'trechos.jsonl'))
print(len(T), 'trechos')
display(pd.read_csv(ROOT / 'base' / 'fontes.csv'))
T.head(6)

31 trechos


,fonte,titulo,url,formato,licenca,n_trechos
0,F1,Carta de Serviços da Prefeitura de Porto Alegr...,https://prefeitura.poa.br/carta-de-servicos/…,HTML (site oficial),Informação pública de órgão oficial; uso com c...,27
1,F2,SP Regula – Entenda os tipos de não conformida...,https://prefeitura.sp.gov.br/web/spregula/w/il...,HTML (site oficial),Informação pública de órgão oficial; uso com c...,4


,id,fonte,ref,texto
0,POA-apagada-descricao,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, serviço de iluminação pública 'L..."
1,POA-apagada-prazo,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, 'Luminária Apagada Durante a Noi..."
2,POA-apagada-canais,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, 'Luminária Apagada Durante a Noi..."
3,POA-oscilante-descricao,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, serviço de iluminação pública 'L..."
4,POA-oscilante-prazo,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, 'Luminária Oscilante'. O prazo d..."
5,POA-oscilante-canais,F1,"Carta de Serviços de Porto Alegre, 'Luminária ...","Porto Alegre, 'Luminária Oscilante'. O serviço..."


## 2. Tarefa e conjunto de teste com gabarito (inclui 'não consta')

In [3]:
G = pd.DataFrame(r.ler_jsonl(ROOT / 'avaliacao' / 'conjunto_teste.jsonl'))
display(G.groupby(['split', 'tipo']).size().unstack())
G[['id','split','tipo','pergunta']].head(8)

tipo,nao_consta,resposta
split,,
dev,3,7
teste,11,19


,id,split,tipo,pergunta
0,q01,teste,resposta,Qual o prazo de atendimento para um poste de i...
1,q02,dev,resposta,"Em Porto Alegre, qual o prazo para trocar uma ..."
2,q03,teste,resposta,Quem é o órgão responsável pela iluminação púb...
3,q04,teste,resposta,Como solicito o conserto de uma luminária fais...
4,q05,dev,resposta,Qual o telefone gratuito para tirar dúvidas so...
5,q06,teste,resposta,Qual o prazo para consertar uma luminária ou b...
6,q07,teste,resposta,Que documentos preciso para pedir um serviço d...
7,q08,dev,resposta,Qual o prazo quando a lâmpada está piscando em...


## 3. Busca: recall@1, recall@3 e limiar (escolhido no conjunto dev)

In [4]:
out = subprocess.run([sys.executable, 'src/01_recuperacao.py'], capture_output=True, text=True, cwd=ROOT); print(out.stdout, out.stderr[-500:])

dev    n= 7  recall@1=0.571  recall@3=1.000
teste  n=19  recall@1=0.632  recall@3=0.947
todos  n=26  recall@1=0.615  recall@3=0.962
limiar escolhido no dev = 0.7003
  dev   : respostas mantidas=1.00  nao_consta barradas=0.00
  teste : respostas mantidas=1.00  nao_consta barradas=0.27
scores top1 dev  resposta : [0.705, 0.755, 0.78, 0.791, 0.82, 0.825, 0.868]
scores top1 dev  nao_consta: [0.726, 0.77, 0.813]

Falhas de recall@3 (para analisar):
  q24 ['POA-faiscando-descricao', 'POA-legislacao', 'POA-oscilante-descricao']
 


## 4. Placar das 3 configurações (conjunto de teste)
Os resultados por pergunta ficam em `resultados/placar_*.jsonl`, gerados por `python src/02_placar.py` (leva de 30 a 60 min em CPU).

In [5]:
out = subprocess.run([sys.executable, 'src/03_resumo.py'], capture_output=True, text=True, cwd=ROOT); print(out.stdout, out.stderr[-500:])

              config  perguntas acertos  acuracia acerto_com_resposta recusa_correta  json_valido  citacao_valida_nas_ok  seg_medio
      A_sem_contexto         30    3/30     0.100                1/19           2/11        0.933                  0.000       12.0
       B_rag_simples         30   25/30     0.833               17/19           8/11        1.000                  0.824        8.5
C_agente_verificador         30   22/30     0.733               12/19          10/11        1.000                  1.000       11.6

ERROS (todas as configurações):
[A_sem_contexto] q01 (resposta) Qual o prazo de atendimento para um poste de iluminação caído em Porto Alegre?
    -> status=ok resp='O prazo de atendimento para um poste de iluminação caído em Porto Alegre é de 24 horas.' cit='Lei 1.124/2004, que trata sobre a segurança pública e a manutenção da infraestrutura urbana.' motivo=None
[A_sem_contexto] q03 (resposta) Quem é o órgão responsável pela iluminação pública em Porto Alegre?
    -

## 5. Demonstração de alucinação: a mesma pergunta nas 3 configurações

In [6]:
idx = r.Indice(); lim = json.load(open(ROOT / 'resultados' / 'limiar.json'))['limiar']
for q in ['Qual o prazo para consertar poste apagado em Fortaleza?', 'Qual o prazo de atendimento para um poste de iluminação caído em Porto Alegre?']:
    print('PERGUNTA:', q)
    for nome, fn in r.CONFIGS.items():
        s = fn(q, idx, lim)
        print(f"  {nome:22s} status={s['status']:10s} resposta={s['resposta'][:90]!r} citacao={s['citacao']!r}")
    print()

PERGUNTA: Qual o prazo para consertar poste apagado em Fortaleza?


  A_sem_contexto         status=ok         resposta='O prazo para consertar um poste apagado em Fortaleza pode variar dependendo da situação e ' citacao='Lei nº 12.527, de 2011, que estabelece normas para a prestação de serviços de energia elétrica.'


  B_rag_simples          status=nao_consta resposta='Não consta na base.' citacao=''
  C_agente_verificador   status=nao_consta resposta='Não consta na base.' citacao=''

PERGUNTA: Qual o prazo de atendimento para um poste de iluminação caído em Porto Alegre?


  A_sem_contexto         status=ok         resposta='O prazo de atendimento para um poste de iluminação caído em Porto Alegre é de 24 horas.' citacao='Lei 1.140/2000, que trata sobre a segurança pública e a manutenção de vias públicas.'


  B_rag_simples          status=ok         resposta='Até 06 horas' citacao='POA-poste-caido-prazo'


  C_agente_verificador   status=ok         resposta='O prazo de atendimento para um poste de iluminação caído em Porto Alegre é de até 06 horas' citacao='POA-poste-caido-prazo'



## 6. Uso integrado (mesma função da API)

In [7]:
s = r.config_C('Como solicito o conserto de uma luminária faiscando em Porto Alegre?', idx, lim)
print(json.dumps(s, ensure_ascii=False, indent=2))

{
  "status": "ok",
  "resposta": "Para solicitar o conserto de uma luminária faiscando em Porto Alegre, você deve entrar em contato com a Secretaria Municipal de Serviços Urbanos (SMSURB). O serviço é fornecido quando há faíscas na ligação entre o fio e a luminária. O responsável pelo conserto é a Secretaria Municipal de Serviços Urbanos (SMSURB).",
  "citacao": "POA-faiscando-descricao",
  "json_valido": true,
  "recuperados": [
    "POA-faiscando-descricao",
    "POA-faiscando-canais",
    "POA-ausente-descricao"
  ],
  "score_top1": 0.8171172142028809,
  "motivo": "verificado",
  "tentativas": 2
}
